# 04 — Nonlinear jump reconstruction and validation

This keeps the original nonlinear case and its existing role in the paper. It adds the missing reference checks, common-observation training, local jump metrics, and independent forward continuation. It also provides a cached nonlinear Tikhonov comparison; the broad QR study is deferred.

A small relative error against temperatures around 0.8–1.0 can hide a large error relative to the **0.2 jump**. Report both global and jump-normalized metrics. Production code below does not insert the true initial profile into any inverse objective.

**Resume revision:** 12 planned neural trainings, four already completed in your upload, **8 remaining**. The verified 1601-node reference is frozen. Classical fits run once per retained observation vector, with each completed alpha cached separately. All selected final neural checkpoints receive cached independent replay refinement.


In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "inverse_study").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Extract the entire study ZIP and open this notebook from its folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from inverse_study.common import *
from inverse_study import numerics as num
from inverse_study import workflows as flow
RESULTS.mkdir(exist_ok=True)
plt.rcParams.update({"font.size": 11, "figure.dpi": 110, "axes.grid": True,
                     "grid.alpha": 0.2, "savefig.bbox": "tight"})
print("Study folder:", ROOT)
from inverse_study import completion as study
from inverse_study import completion_checks as checks
REDUCED = RESULTS / "reduced_plan"
REDUCED.mkdir(parents=True, exist_ok=True)
study.verify_protocol_lock(require_reference=True)


In [ ]:
p = problem_spec("nonlinear1d")
reference, reference_gate = flow.load_reference(require_pass=True)
folder = freeze_observations(p, obs_per_dim=257, noise_level=.05, noise_seed=3100, reference=reference)
_, obs, meta = load_observations(folder)
display(pd.Series(meta).to_frame("value"))
print("Reference numerical floor:", reference_gate["suggested_numerical_floor"])
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(reference["x"], initial_jump(reference["x"]), "k--", label="Initial reference: evaluation only")
ax.plot(reference["x"], reference["terminal"], label="Clean terminal reference")
ax.scatter(obs["X_obs"][:,0], obs["y_obs"], s=8, alpha=.5, label="Frozen terminal observations")
ax.set(xlabel="x", ylabel="Temperature"); ax.legend(); plt.show()

## 1. Recompute local metrics from the original archived fields

These are postprocessed archived reconstructions, not corrected-protocol runs. Plateau values are measured on x∈[0.25,0.40] and [0.60,0.75]. Width uses crossings of the true 10%/90% levels 0.82 and 0.98 near the jump. A missing crossing produces an unavailable value rather than a fabricated width.

In [ ]:
legacy_rows = []
for d in sorted((DATA / "legacy_runs").glob("discontinuous*")):
    with np.load(d / "predictions.npz", allow_pickle=False) as z:
        xx = z["X_space"].ravel(); uu = z["theta0_pred"].ravel()
    legacy_rows.append({"run_id": d.name, **jump_metrics(xx,uu), "protocol": "legacy_postprocessing"})
legacy_jump = pd.DataFrame(legacy_rows)
display(legacy_jump)
(RESULTS / "nonlinear").mkdir(exist_ok=True)
legacy_jump.to_csv(REDUCED / "legacy_jump_metrics.csv", index=False)

## 2. Complete the reduced neural cohort

The plan has plain and adaptive-Fourier methods, clean and 5% noise conditions, and three repetitions: 12 jobs. For noisy data, use only the three predetermined paired noise/training settings. Four matching screening jobs are already complete and are skipped automatically.


In [ ]:
RUN_NEURAL = True
MAX_NEW_JOBS = None
plan = study.compact_nonlinear_plan()
plan.to_csv(REDUCED / "nonlinear_plan.csv",index=False)
inventory = study.plan_inventory(plan)
display(inventory[["method","noise_level","noise_seed","training_seed","status","reason"]])
print("Already completed:",int(inventory.status.eq("completed").sum()),
      "| pending:",int(inventory.status.isin(["missing","resumable"]).sum()))
if RUN_NEURAL:
    nonlinear_runs = study.run_missing(plan, max_new_jobs=MAX_NEW_JOBS)
    display(nonlinear_runs)
else:
    print("Neural training is OFF; completed runs remain available.")


## 3. Classical comparator on all four retained observation vectors

The piecewise-linear initial field has 17 knots. Its interior values fit the same terminal point data through the independent nonlinear forward solver, with a curvature penalty. Initial truth is used only for post-selection error metrics and a separate synthetic forward-discretization calibration.

This section evaluates clean data and the three noisy vectors **once each**, using the same declared 13-alpha grid. It saves results by dataset and configuration, avoiding overwriting one noise realization with another. Each alpha starts from the same data-derived initial guess, exactly as in the original numerical routine. An interrupted parameter sweep resumes from its uncompleted alphas.

A cached optimizer failure or an unattainable discrepancy remains a reported outcome. The code does not silently discard it, choose the smallest initial-state error, or certify publication readiness. Knot/mesh sensitivity and an adequate alpha bracket still need inspection before final comparative claims.


In [ ]:
RUN_NONLINEAR_BASELINE = True
MAX_NEW_BASELINE_CASES = None
INVERSE_NX = 201
N_KNOTS = 17
ALPHAS = np.logspace(-9,-1,13)
baseline_inventory = checks.nonlinear_baseline_inventory(
    plan, nx=INVERSE_NX, n_knots=N_KNOTS, alphas=ALPHAS)
display(baseline_inventory)
if RUN_NONLINEAR_BASELINE:
    baseline_results = checks.run_nonlinear_baselines(
        plan, max_new_cases=MAX_NEW_BASELINE_CASES,
        nx=INVERSE_NX, n_knots=N_KNOTS, alphas=ALPHAS)
    display(baseline_results)
else:
    print("Classical optimization is OFF; the inventory above does not fit a model.")


## 4. Deferred QR audit

The primary nonlinear completion comparison uses the point-data Tikhonov method above. The broad QR closure/gamma/mesh/time audit is **OFF** and is excluded from the reduced completion plan.

The original QR sources and formulation diagnostics remain in the bundle. A future audit needs its own predeclared closure, interpolation floor, and mesh/time study. The old QR results remain provisional; this revision does not validate or relabel them.


In [ ]:
RUN_QR_STUDY = False
if RUN_QR_STUDY:
    raise RuntimeError(
        "The broad QR study is deferred in this completion plan. "
        "Use the validated point-observation baseline above. "
        "A separate QR audit must predeclare closure, interpolation floor, and per-job caching.")
else:
    print("QR sweep deferred. Existing QR sources and diagnostics are preserved.")


## 5. Independent replay refinement for every completed selected model

Use the same final saved checkpoint on 401, 801 and 1601 nodes. Each grid result is cached; repeated notebook execution does not repeat completed forward solves. This check writes under `results/reduced_validation`, preserving the original `evaluation.json` files.

The empirical acceptance threshold is the larger of `1e-4 * ||y||`, 5% of the known synthetic noise norm, and a small roundoff floor. It measures discretization consistency only. A small refined terminal residual does not establish accurate recovery of the initial jump, and a large boundary mismatch remains visible in the report.


In [ ]:
RUN_REPLAY_REFINEMENT = True
MAX_NEW_REPLAYS = None
if RUN_REPLAY_REFINEMENT:
    replay_summary = checks.refine_neural_replays(plan, max_new_runs=MAX_NEW_REPLAYS)
    display(replay_summary)
else:
    print("Replay refinement is OFF; no neural model is retrained by this section.")
